In [4]:
import pandas as pd
import sqlite3
from datetime import datetime

print("=" * 70)
print("STEP 1: LOAD CLEANED DATA")
print("=" * 70)

# Load the cleaned data we created earlier
df = pd.read_csv('data/cleaned_properties.csv')

print(f"Loaded {len(df)} properties")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst few rows:")
print(df.head())

# ============================================
# STEP 2: CREATE DATABASE CONNECTION
# ============================================

print("\n" + "=" * 70)
print("STEP 2: CREATE SQL DATABASE")
print("=" * 70)

# Create SQLite database (creates file: real_estate.db)
conn = sqlite3.connect('data/real_estate.db')
cursor = conn.cursor()

print("✓ Connected to SQLite database")
print("✓ Database file: data/real_estate.db")

# ============================================
# STEP 3: CREATE TABLE
# ============================================

print("\n" + "=" * 70)
print("STEP 3: CREATE PROPERTIES TABLE")
print("=" * 70)

# Drop table if exists (for re-running)
cursor.execute('DROP TABLE IF EXISTS properties')

# Create table with proper data types
create_table = '''
CREATE TABLE properties (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price INTEGER,
    price_per_m2 REAL,
    location TEXT,
    district TEXT,
    area INTEGER,
    bedrooms INTEGER,
    property_type TEXT,
    condition TEXT,
    has_parking BOOLEAN,
    has_garden BOOLEAN,
    date_listed DATE,
    date_scraped DATE,
    days_listed INTEGER,
    price_category TEXT,
    quality_score INTEGER,
    features_score INTEGER,
    value_score REAL,
    market_type TEXT
)
'''

cursor.execute(create_table)
conn.commit()

print("✓ Created 'properties' table")
print(f"✓ Columns: 20 fields")

# ============================================
# STEP 4: LOAD DATA INTO TABLE
# ============================================

print("\n" + "=" * 70)
print("STEP 4: LOAD DATA INTO DATABASE")
print("=" * 70)

# Insert cleaned data into database
df.to_sql('properties', conn, if_exists='replace', index=False)

# Verify data was loaded
cursor.execute('SELECT COUNT(*) FROM properties')
count = cursor.fetchone()[0]

print(f"✓ Loaded {count} properties into database")

# ============================================
# STEP 5: CREATE INDEXES (for faster queries)
# ============================================

print("\n" + "=" * 70)
print("STEP 5: CREATE INDEXES")
print("=" * 70)

# Indexes make queries faster
indexes = [
    'CREATE INDEX idx_district ON properties(district)',
    'CREATE INDEX idx_price ON properties(price)',
    'CREATE INDEX idx_property_type ON properties(property_type)',
    'CREATE INDEX idx_price_category ON properties(price_category)',
    'CREATE INDEX idx_market_type ON properties(market_type)',
]

for index in indexes:
    cursor.execute(index)
    conn.commit()

print("✓ Created 5 indexes for faster queries")

# ============================================
# STEP 6: RUN USEFUL QUERIES
# ============================================

print("\n" + "=" * 70)
print("STEP 6: ANALYSIS QUERIES")
print("=" * 70)

# Query 1: Average price by district
print("\n📍 QUERY 1: Average Price by District")
print("-" * 70)

query1 = '''
SELECT 
    district,
    COUNT(*) as total_properties,
    ROUND(AVG(price), 0) as avg_price,
    MIN(price) as min_price,
    MAX(price) as max_price
FROM properties
GROUP BY district
ORDER BY avg_price DESC
LIMIT 10
'''

result1 = pd.read_sql_query(query1, conn)
print(result1)

# Query 2: Price per m² by property type
print("\n🏠 QUERY 2: Price per m² by Property Type")
print("-" * 70)

query2 = '''
SELECT 
    property_type,
    COUNT(*) as count,
    ROUND(AVG(price_per_m2), 2) as avg_price_per_m2,
    ROUND(MIN(price_per_m2), 2) as min_price_per_m2,
    ROUND(MAX(price_per_m2), 2) as max_price_per_m2
FROM properties
WHERE property_type != 'Land'
GROUP BY property_type
ORDER BY avg_price_per_m2 DESC
'''

result2 = pd.read_sql_query(query2, conn)
print(result2)

# Query 3: Best value properties (low price_per_m2)
print("\n💎 QUERY 3: Top 10 Best Value Properties")
print("-" * 70)

query3 = '''
SELECT 
    id,
    title,
    price,
    area,
    ROUND(price_per_m2, 2) as price_per_m2,
    district
FROM properties
WHERE property_type != 'Land'
ORDER BY price_per_m2 ASC
LIMIT 10
'''

result3 = pd.read_sql_query(query3, conn)
print(result3)

# Query 4: Most expensive properties
print("\n👑 QUERY 4: Top 10 Most Expensive Properties")
print("-" * 70)

query4 = '''
SELECT 
    id,
    title,
    price,
    area,
    bedrooms,
    property_type,
    district
FROM properties
ORDER BY price DESC
LIMIT 10
'''

result4 = pd.read_sql_query(query4, conn)
print(result4)

# Query 5: Properties by market type
print("\n🌍 QUERY 5: Properties by Market Type")
print("-" * 70)

query5 = '''
SELECT 
    market_type,
    COUNT(*) as count,
    ROUND(AVG(price), 0) as avg_price,
    ROUND(AVG(price_per_m2), 2) as avg_price_per_m2
FROM properties
GROUP BY market_type
ORDER BY avg_price DESC
'''

result5 = pd.read_sql_query(query5, conn)
print(result5)

# Query 6: Recent listings
print("\n📅 QUERY 6: Most Recent Listings")
print("-" * 70)

query6 = '''
SELECT 
    id,
    title,
    price,
    days_listed,
    property_type,
    district
FROM properties
ORDER BY days_listed ASC
LIMIT 10
'''

result6 = pd.read_sql_query(query6, conn)
print(result6)

# Query 7: Investment opportunities (good value, not too old)
print("\n🎯 QUERY 7: Investment Opportunities")
print("   (Low price_per_m2 + Recently listed)")
print("-" * 70)

query7 = '''
SELECT 
    id,
    title,
    price,
    area,
    ROUND(price_per_m2, 2) as price_per_m2,
    days_listed,
    district,
    value_score
FROM properties
WHERE property_type NOT IN ('Land')
  AND days_listed <= 30
  AND value_score >= 75
ORDER BY value_score DESC
LIMIT 15
'''

result7 = pd.read_sql_query(query7, conn)
print(result7)

# ============================================
# STEP 7: SAVE QUERY RESULTS TO CSV
# ============================================

print("\n" + "=" * 70)
print("STEP 7: EXPORT RESULTS")
print("=" * 70)

# Save each query result as CSV for Power BI
result1.to_csv('data/query_price_by_district.csv', index=False)
result2.to_csv('data/query_price_by_type.csv', index=False)
result3.to_csv('data/query_best_value.csv', index=False)
result4.to_csv('data/query_most_expensive.csv', index=False)
result5.to_csv('data/query_by_market_type.csv', index=False)
result7.to_csv('data/query_investments.csv', index=False)

print("✓ Saved query results:")
print("  - query_price_by_district.csv")
print("  - query_price_by_type.csv")
print("  - query_best_value.csv")
print("  - query_most_expensive.csv")
print("  - query_by_market_type.csv")
print("  - query_investments.csv")

# ============================================
# STEP 8: DATABASE INFO
# ============================================

print("\n" + "=" * 70)
print("STEP 8: DATABASE SUMMARY")
print("=" * 70)

# Total statistics
cursor.execute('SELECT COUNT(*) FROM properties')
total = cursor.fetchone()[0]

cursor.execute('SELECT SUM(price) FROM properties')
total_value = cursor.fetchone()[0]

cursor.execute('SELECT AVG(price) FROM properties')
avg_price = cursor.fetchone()[0]

print(f"\n📊 DATABASE STATS:")
print(f"  Total properties: {total}")
print(f"  Total market value: {total_value:,.0f} TND")
print(f"  Average price: {avg_price:,.0f} TND")
print(f"  Database file: data/real_estate.db")

print("\n✅ SQL DATABASE SETUP COMPLETE!")
print("\nYou can now:")
print("  - Query data directly from database")
print("  - Connect to Power BI for visualization")
print("  - Run more complex queries")

# Close connection
conn.close()
print("\n✓ Database connection closed")

STEP 1: LOAD CLEANED DATA
Loaded 170 properties

Columns: ['id', 'title', 'price', 'price_per_m2', 'location', 'district', 'area', 'bedrooms', 'property_type', 'condition', 'has_parking', 'has_garden', 'date_listed', 'date_scraped', 'days_listed', 'price_category', 'quality_score', 'features_score', 'value_score', 'market_type']

First few rows:
   id                              title   price  price_per_m2  \
0   3            Villa 3P 319m² - Mahdia  871935       2733.34   
1   4         Duplex 4P 93m² - Ben Arous   59841        643.45   
2   5         Villa 6P 184m² - Kasserine  272586       1481.45   
3   6             Duplex 3P 93m² - Gafsa   45887        493.41   
4   8  Apartment 3P 102m² - Tunis Centre  239360       2346.67   

                 location      district  area  bedrooms property_type  \
0        Mahdia - Tunisia        Mahdia   319         3         Villa   
1     Ben Arous - Tunisia     Ben Arous    93         4        Duplex   
2     Kasserine - Tunisia     Kasser